# XCOMPS new version

## 1. Analysis 

In [1]:
import json
import urllib.request
import re
from difflib import SequenceMatcher
import pandas as pd

_DATA_URL = "https://raw.githubusercontent.com/LinyangHe/XCOMPS/main/data/comps_nl.json"

def load_xcomps_nl() -> list[dict]:
	with urllib.request.urlopen(_DATA_URL) as response:
		data = json.load(response)
	return [
		{
			"sentence_a": row["acceptable_sent"],
			"sentence_b": row["unacceptable_sent"],
			"negative_sample_type": row.get("negative_sample_type"),
		}
		for row in data.values()
	]

def load_raw_xcomps_nl(url):
	with urllib.request.urlopen(url) as response:
		data = json.load(response)
	
	return data

In [2]:
xcomps_data = load_raw_xcomps_nl(_DATA_URL)

In [3]:
xcomps_df = pd.DataFrame.from_dict(xcomps_data, orient='index')
xcomps_df

,lang,property,acceptable_concept,unacceptable_concept,acceptable_sent,unacceptable_sent,negative_sample_type,similarity
1,nl,absorbeert zweet,sok,kous,Sok absorbeert zweet.,Kous absorbeert zweet.,taxonomic,0.833333
2,nl,absorbeert zweet,sok,trui,Sok absorbeert zweet.,Trui absorbeert zweet.,overlap,0.480000
3,nl,absorbeert zweet,sok,pantybroek,Sok absorbeert zweet.,Pantybroek absorbeert zweet.,co-occurrence,0.556055
4,nl,absorbeert zweet,sok,vat,Sok absorbeert zweet.,Vat absorbeert zweet.,random,0.000000
5,nl,absorbeert water,rijst,maïs,Rijst absorbeert water.,Maïs absorbeert water.,taxonomic,0.777778
...,...,...,...,...,...,...,...,...
49334,nl,draagt schoenen,paard,ezel,Paard draagt schoenen.,Ezel draagt schoenen.,overlap,0.633027
49335,nl,draagt schoenen,paard,hond,Paard draagt schoenen.,Een hond draagt schoenen.,co-occurrence,0.624628
49336,nl,draagt schoenen,paard,wolf,Een paard draagt schoenen.,Wolf draagt schoenen.,random,0.000000
8177,nl,kan worden gebruikt voor opslag,fles,lepel,Een fles kan worden gebruikt voor opslag.,Een pollepel kan worden gebruikt voor opslag.,taxonomic,0.800000


In [4]:
LEAK_RE = re.compile(r"\t|\b(concept|property|sentence)\s*:", re.IGNORECASE)
# French / German / English function words that are not Dutch words
FOREIGN_RE = re.compile(
	r"\b(peut|peuvent|être|une|un|les|dans|avec|plusieurs|se|a|à|kann|ein|eine|the|can)\b", re.IGNORECASE
)
ARTICLES = {"een", "de", "het"}


def norm(text: str) -> str:
	return re.sub(r"[^\w\s-]", "", text.lower()).strip()


def is_word_form(concept: str, word: str) -> bool:
	concept, word = norm(concept), norm(word)
	if len(word) >= 3 and (concept.startswith(word) or word.startswith(concept)):
		return True
	return SequenceMatcher(None, concept, word).ratio() >= 0.7


def classify_missing(row: dict, role: str) -> str:
	concept, sent, prop = row[f"{role}_concept"], row[f"{role}_sent"], row["property"]
	words = norm(sent).split()
	if LEAK_RE.search(sent):
		return "leaked_output"
	if FOREIGN_RE.search(sent):
		return "not_dutch"
	if words[0] == norm(prop).split()[0]:
		return "concept_dropped"
	# compare the concept with the subject: the first two non-article words
	subject = [w for w in words if w not in ARTICLES][:2]
	if any(is_word_form(concept, w) for w in subject):
		return "different_word_form"
	return "different_word"


# every sentence that does not contain its concept as written
missing_df = pd.DataFrame([
	{
		"pair_id": pair_id,
		"role": role,
		"category": classify_missing(row, role),
		"concept": row[f"{role}_concept"],
		"sentence": row[f"{role}_sent"],
		"property": row["property"],
	}
	for pair_id, row in xcomps_data.items()
	for role in ("acceptable", "unacceptable")
	if row[f"{role}_concept"].lower() not in row[f"{role}_sent"].lower()
])

display(pd.crosstab(missing_df["category"], missing_df["role"], margins=True))

with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
	for category in ("different_word_form", "different_word", "not_dutch", "leaked_output"):
		print(f"--- {category} ---")
		display(missing_df[missing_df["category"] == category].drop(columns="category"))

role,acceptable,unacceptable,All
category,,,
concept_dropped,185,238,423
different_word,18,11,29
different_word_form,20,24,44
leaked_output,0,1,1
not_dutch,2,10,12
All,225,284,509


--- different_word_form ---


,pair_id,role,concept,sentence,property
123,3404,unacceptable,snelboot,Een speedboot is te vinden in dierentuinen.,kan in dierentuinen worden gevonden
128,3534,unacceptable,lippen,Een lip kan op de armen worden gevonden.,kan op de armen worden gevonden
199,7593,unacceptable,gewaden,Een gewaad kan om de taille worden gebonden.,kan om de taille worden gebonden
211,8077,acceptable,dobbelstenen,Een dobbelsteen kan voor gokken worden gebruikt.,kan voor gokken worden gebruikt
273,9697,unacceptable,kwallen,Een kwal kan bloeden.,kan bloeden
278,10136,unacceptable,poppen,Een pop kan raketten dragen.,kan raketten dragen
354,16811,unacceptable,kwallen,Een kwal kan inkt spuiten.,kan inkt spuiten
362,17932,unacceptable,poppen,Een pop komt in smaken.,komt in smaken
363,18988,acceptable,dobbelstenen,Een dobbelsteen toont nummers aan.,toont nummers aan
370,20943,unacceptable,kangaroo,Een kangoeroe heeft een eendrekening.,heeft een eendenbek


--- different_word ---


,pair_id,role,concept,sentence,property
154,4051,acceptable,broodje,Een sandwich kan worden gegrild.,kan gegrild worden
289,11940,unacceptable,deegroller,Een rollende pen kan in de zak passen.,past in de zak
325,13732,unacceptable,prei,Leek kan sissen.,kan sissen
329,14239,unacceptable,deegroller,Een rollende pen kan baby's veilig houden.,kan baby's veilig houden
360,17320,unacceptable,badpak,Een zwempak kan vertrappen.,kan vertrappen
364,19065,acceptable,badpak,Een zwempak droogt snel op.,droogt snel
365,19066,acceptable,badpak,Een zwempak droogt snel op.,droogt snel
366,19067,acceptable,badpak,Een zwempak droogt snel op.,droogt snel
367,19068,acceptable,badpak,Een zwempak droogt snel op.,droogt snel
368,19144,unacceptable,badpak,Een zwempak eet vuil.,eet vuil


--- not_dutch ---


,pair_id,role,concept,sentence,property
28,1600,unacceptable,snelboot,Bateau à moteur peut être mangé sec.,kan gedroogd gegeten worden
48,2300,unacceptable,kettingzaag,Tronçonneuse peut être trouvé près des piscines.,kan worden gevonden bij zwembaden
200,7617,unacceptable,vleermuis,Peut être entraîné.,kan getraind worden
206,7907,unacceptable,gitaar,Peut être utilisé par les forces armées.,kan door strijdkrachten worden gebruikt
222,8614,unacceptable,broodrooster,Grille-pain peut être utilisé pour communiquer.,kan worden gebruikt om te communiceren
224,8708,unacceptable,bromfiets,Une mobylette peut être utilisé pour remplir des piscines.,kan worden gebruikt om zwembaden te vullen
295,12520,unacceptable,bromfiets,Une mobylette peut pendre du plafond.,kan aan het plafond hangen
384,28934,unacceptable,snelboot,Un bateau à moteur a plusieurs parties.,heeft meerdere delen
385,29077,acceptable,driewieler,Un tricycle a une roue à l'avant et deux à l'arrière.,heeft één wiel aan de voorkant en twee aan de achterkant
386,29078,acceptable,driewieler,Un tricycle a une roue à l'avant et deux à l'arrière.,heeft één wiel aan de voorkant en twee aan de achterkant


--- leaked_output ---


,pair_id,role,concept,sentence,property
195,6955,unacceptable,kip,German concept: Huhn\tGerman property: kann in Kartons verkauft werden\tGerman sentence: Ein Huhn kann in Kartons verkauft werden.,kan in kartons verkocht worden


In [4]:
from HanTa import HanoverTagger as ht

tagger_nl = ht.HanoverTagger('morphmodel_dutch.pgz')
tagger_nl.tag_word('woont')

[('WW(pv,tgw,met-t)', -9.648586419721996)]

In [5]:
xcomps_df.loc[xcomps_df['acceptable_concept'] == "tonijn"]

,lang,property,acceptable_concept,unacceptable_concept,acceptable_sent,unacceptable_sent,negative_sample_type,similarity
593,nl,kan worden geslagen,tonijn,Makreel,Tonijn kan worden geslagen.,Makreel kan worden geslagen.,taxonomic,0.888889
594,nl,kan worden geslagen,tonijn,Haring,Tonijn kan worden geslagen.,Kan worden geslagen,overlap,0.766234
595,nl,kan worden geslagen,tonijn,garnalen,Tonijn kan worden geslagen.,Garnalen kan worden geslagen.,co-occurrence,0.723314
596,nl,kan worden geslagen,tonijn,kano,Tonijn kan worden geslagen.,Kano kan worden geslagen.,random,0.000000
33209,nl,is een vis,tonijn,kikker,Tonijn is een vis.,Een kikker is een vis.,taxonomic,0.600000
33210,nl,is een vis,tonijn,kreeft,Tonijn is een vis.,Kreeft is een vis.,overlap,0.521739
33211,nl,is een vis,tonijn,garnalen,Tonijn is een vis.,Garnalen is een vis.,co-occurrence,0.723314
33212,nl,is een vis,tonijn,beer,Tonijn is een vis.,Beer is een vis.,random,0.000000
38009,nl,wordt als steak gegeten,tonijn,Makreel,Tonijn wordt als steak gegeten.,Makreel wordt als steak gegeten.,taxonomic,0.888889
38010,nl,wordt als steak gegeten,tonijn,Haring,Tonijn wordt als steak gegeten.,Een haring wordt als steak gegeten.,overlap,0.766234


In [6]:
xcomps_df.loc[xcomps_df['property'] == "wordt geserveerd met vlees en aardappelen"]

,lang,property,acceptable_concept,unacceptable_concept,acceptable_sent,unacceptable_sent,negative_sample_type,similarity
43081,nl,wordt geserveerd met vlees en aardappelen,broccoli,kool,Broccoli wordt geserveerd met vlees en aardapp...,Kool wordt geserveerd met vlees en aardappelen.,taxonomic,0.800000
43082,nl,wordt geserveerd met vlees en aardappelen,broccoli,spinazie,Broccoli wordt geserveerd met vlees en aardapp...,Spinazie wordt geserveerd met vlees en aardapp...,overlap,0.727273
43083,nl,wordt geserveerd met vlees en aardappelen,broccoli,asperge,Broccoli wordt geserveerd met vlees en aardapp...,Asperge wordt geserveerd met vlees en aardappe...,co-occurrence,0.831903
43084,nl,wordt geserveerd met vlees en aardappelen,broccoli,chocolade,Broccoli wordt geserveerd met vlees en aardapp...,Chocolade wordt geserveerd met vlees en aardap...,random,0.000000


In [7]:
comps_df = pd.read_json('data/comps_base.jsonl', lines=True)

In [8]:
pd.set_option('display.max_rows', 208)
# comps_df.loc[comps_df['acceptable_concept'] == "cricket"]

## 2. Build new version

In [9]:
import pandas as pd

In [11]:
concepts_analysed = pd.read_excel('data/NL_XCOMPS/concepts.xlsx', index_col='Column1').rename_axis('index')
concepts_analysed['update'] = concepts_analysed['update'].fillna('no')
concepts_analysed['type'] = concepts_analysed['type'].fillna('sg')

In [12]:
properties_analysed = pd.read_excel('data/NL_XCOMPS/properties_with_concepts.xlsx', index_col='index')
properties_analysed['correct'] = properties_analysed['correct'].fillna('yes')
properties_analysed['update'] = properties_analysed['update'].fillna('nvt')
properties_analysed

,acceptable_concept,property,correct,update
index,,,,
0,portemonnee,kan foto's bevatten,yes,nvt
1,wilg,heeft hangende takken,yes,nvt
2,onderzeeboot,is elliptisch van vorm,yes,nvt
3,grapefruit,wordt als ingrediënt in recepten gebruikt,yes,nvt
4,pinguïn,woont in Antarctica,yes,nvt
...,...,...,...,...
3538,sluier,wordt gebruikt om te verbergen,not correct,wordt gebruikt om je gezicht mee te verbergen
3539,vlot,kan zinken,yes,nvt
3540,limoen,is pittig,not correct,is scherp


In [13]:
import re

# --- lookups ---
c = concepts_analysed.set_index('concepts')
concept_fix  = c['update'].where(c['update'] != 'no', c.index.to_series()).to_dict()  # old -> new concept
concept_type = c['type'].to_dict()                                                     # old concept -> sg/pl/mass/sg/mass

# one row per concept: 'schaap|varken' shares a property -> two rows
p = properties_analysed.assign(acceptable_concept=properties_analysed['acceptable_concept'].str.split('|')).explode('acceptable_concept')
p['acceptable_concept'] = p['acceptable_concept'].str.strip()
p = p.set_index(['acceptable_concept', 'property'])
assert not p.index.duplicated().any()

status = p['correct'].str.strip()
prop_fix = p.loc[p['update'] != 'nvt', 'update'].to_dict()
to_drop  = set(p.index[(status == 'dubieus') | ((status != 'yes') & (p['update'] == 'nvt'))])  # dubieus + flagged without update

ARTICLE = {'sg': 'een ', 'pl': '', 'mass': ''}

# lowercase, strip punctuation (keep in-word ' and -, e.g. foto's, kant-en-klaar)
PUNCT = re.compile(r"[^\w\s'-]|(?<!\w)['-]|['-](?!\w)")
def normalize(s):
    return s.str.lower().str.replace(PUNCT, '', regex=True).str.split().str.join(' ')

# --- apply ---
df = xcomps_df.copy()
keys = pd.Series(list(zip(df['acceptable_concept'], df['property'])), index=df.index)

df = df[~keys.isin(to_drop)]
keys = keys[df.index]

not_in_sheet = df.index[~keys.isin(p.index)].tolist()

types = df[['acceptable_concept', 'unacceptable_concept']].apply(lambda s: s.map(concept_type))
sg_mass = df.index[(types == 'sg/mass').any(axis=1)].tolist()
todo = ~df.index.isin(sg_mass)  # sg/mass rows are left untouched for manual editing

df.loc[todo, 'property'] = [prop_fix.get(k, k[1]) for k in keys[todo]]
for side in ['acceptable', 'unacceptable']:
    col = f'{side}_concept'
    article = types.loc[todo, col].map(ARTICLE)  # type of the original concept, before renaming
    df.loc[todo, col] = df.loc[todo, col].map(concept_fix)
    df.loc[todo, f'{side}_sent'] = article + df.loc[todo, col] + ' ' + df.loc[todo, 'property']

# --- normalize ---
text_cols = ['property', 'acceptable_concept', 'unacceptable_concept', 'acceptable_sent', 'unacceptable_sent']
df[text_cols] = df[text_cols].apply(normalize)
df[['acceptable_sent', 'unacceptable_sent']] += '.'

print(f'dropped {len(xcomps_df) - len(df)} dubieus / flagged-without-update rows')
print(f'not in properties sheet: {len(not_in_sheet)} | sg/mass: {len(sg_mass)}')

dropped 308 dubieus / flagged-without-update rows
not in properties sheet: 4 | sg/mass: 1354


In [14]:
from pathlib import Path

# sg/mass rows for manual editing: concepts and property with corrections applied, sentences as in the original
orig = xcomps_df.loc[sg_mass]
manual = pd.DataFrame({
    'acceptable_concept':   orig['acceptable_concept'].map(concept_fix),
    'acceptable_type':      types.loc[sg_mass, 'acceptable_concept'],
    'unacceptable_concept': orig['unacceptable_concept'].map(concept_fix),
    'unacceptable_type':    types.loc[sg_mass, 'unacceptable_concept'],
    'property':             [prop_fix.get(k, k[1]) for k in keys[sg_mass]],
    'acceptable_sent':      df.loc[sg_mass, 'acceptable_sent'],
    'unacceptable_sent':    df.loc[sg_mass, 'unacceptable_sent'],
})
manual[['acceptable_concept', 'unacceptable_concept', 'property']] = manual[['acceptable_concept', 'unacceptable_concept', 'property']].apply(normalize)

# never overwrite the manually edited file
if Path('data/NL_XCOMPS/sg_mass_manual.xlsx').exists():
    print('data/NL_XCOMPS/sg_mass_manual.xlsx already exists, not overwriting')
else:
    manual.to_excel('data/NL_XCOMPS/sg_mass_manual.xlsx', index_label='index')
manual

data/NL_XCOMPS/sg_mass_manual.xlsx already exists, not overwriting


,acceptable_concept,acceptable_type,unacceptable_concept,unacceptable_type,property,acceptable_sent,unacceptable_sent
6,rijst,mass,bloemkool,sg/mass,absorbeert water,rijst absorbeert water.,bloemkool absorbeert water.
7,rijst,mass,aardappel,sg/mass,absorbeert water,rijst absorbeert water.,aardappel absorbeert water.
75,geranium,sg,eucalyptus,sg/mass,trekt bijen aan,geranium trekt bijen aan.,eucalyptus trekt bijen aan.
106,prei,sg,kool,sg/mass,behoort tot de uienfamilie,prei behoort tot de uienfamilie.,kool behoort tot de uienfamilie.
145,ananas,sg/mass,krent,sg,verandert in de loop van de tijd,ananas verandert in de loop van de tijd.,krent verandert in de loop van de tijd.
...,...,...,...,...,...,...,...
49028,haar,sg/mass,gorilla,sg,verzendt berichten naar de hersenen,haar verzendt berichten naar de hersenen.,een gorilla verzendt berichten naar de hersenen.
49184,magnetron,sg,kabeljauw,sg/mass,maakt gebruik van microgolfstraling,magnetron maakt gebruik van microgolfstraling.,kabeljauw maakt gebruik van microgolfstraling.
49237,haan,sg,kalkoen,sg/mass,staat vroeg op,een haan staat vroeg op.,kalkoen staat vroeg op.
49240,haan,sg,kaas,sg/mass,staat vroeg op,een haan staat vroeg op.,kaas staat vroeg op.


In [15]:
# --- merge manual sg/mass decisions back into df ---
edited = pd.read_excel('data/NL_XCOMPS/sg_mass_manual.xlsx', index_col='index', dtype={'index': str})
type_cols = ['acceptable_type', 'unacceptable_type']
edited[type_cols] = edited[type_cols].replace({'g': 'sg'})  # typo in 6 rows
assert edited[type_cols].isin(ARTICLE.keys()).all().all(), edited[type_cols].stack().value_counts()
assert edited.index.isin(df.index).all()

edited[['acceptable_concept', 'unacceptable_concept', 'property']] = edited[['acceptable_concept', 'unacceptable_concept', 'property']].apply(normalize)
df.loc[edited.index, 'property'] = edited['property']
for side in ['acceptable', 'unacceptable']:
    df.loc[edited.index, f'{side}_concept'] = edited[f'{side}_concept']
    df.loc[edited.index, f'{side}_sent'] = (edited[f'{side}_type'].map(ARTICLE) + edited[f'{side}_concept'] + ' ' + edited['property'] + '.')

print(f'merged {len(edited)} manually typed rows')

merged 1354 manually typed rows


In [16]:
# --- duplicate sentence pairs ---
pair = ['acceptable_sent', 'unacceptable_sent']
dups = df[df.duplicated(pair, keep=False)].sort_values(pair)
print(f'{len(dups)} rows in {dups.groupby(pair).ngroups} duplicate groups ({df.duplicated(pair).sum()} extra copies)')

df = df.drop_duplicates(pair, keep='first')

193 rows in 94 duplicate groups (99 extra copies)


In [17]:
df_deduped = df.copy()
df_deduped = df_deduped[df_deduped['acceptable_concept'] != df_deduped['unacceptable_concept']]
print(f'Dropped {len(df) - len(df_deduped)} where acceptable concept == unacceptable')

Dropped 108 where acceptable concept == unacceptable


In [19]:
cols = ['acceptable_concept', 'unacceptable_concept']
contains = [a != b and (a in b or b in a) for a, b in zip(df['acceptable_concept'], df['unacceptable_concept'])]
overlap_rows = df[contains].sort_values(cols)

# never overwrite the manually edited file
if Path('data/NL_XCOMPS/overlap.xlsx').exists():
    print('data/NL_XCOMPS/overlap.xlsx already exists, not overwriting')
else:
    overlap_rows.to_excel('overlap.xlsx', index_label='index')

data/NL_XCOMPS/overlap.xlsx already exists, not overwriting


In [21]:
# --- drop overlapping concept pairs marked keep == 'no' in overlap.xlsx ---
overlap_checked = pd.read_excel('data/NL_XCOMPS/overlap.xlsx', index_col='Kolom1', dtype={'Kolom1': str})
keep = overlap_checked['keep'].str.strip().str.lower()
assert keep.isin(['yes', 'no']).all(), keep.value_counts(dropna=False)
assert overlap_checked.index.isin(df_deduped.index).all()

drop_overlap = overlap_checked.index[keep == 'no']
df_deduped = df_deduped.drop(drop_overlap)
print(f'Dropped {len(drop_overlap)} overlapping rows marked "no", {len(df_deduped)} rows left')

Dropped 191 overlapping rows marked "no", 13662 rows left


In [22]:
if not Path('data/NL_XCOMPS/deduped_NL_xcomps.xlsx').exists():
	df_deduped.to_excel('data/NL_XCOMPS/deduped_NL_xcomps.xlsx')

## 3. Make kid slice

In [25]:
df_deduped.head(20)

,lang,property,acceptable_concept,unacceptable_concept,acceptable_sent,unacceptable_sent,negative_sample_type,similarity
1,nl,absorbeert zweet,sok,kous,een sok absorbeert zweet.,een kous absorbeert zweet.,taxonomic,0.833333
2,nl,absorbeert zweet,sok,trui,een sok absorbeert zweet.,een trui absorbeert zweet.,overlap,0.480000
3,nl,absorbeert zweet,sok,pantybroek,een sok absorbeert zweet.,een pantybroek absorbeert zweet.,co-occurrence,0.556055
4,nl,absorbeert zweet,sok,vat,een sok absorbeert zweet.,een vat absorbeert zweet.,random,0.000000
5,nl,absorbeert water,rijst,maïs,rijst absorbeert water.,maïs absorbeert water.,taxonomic,0.777778
6,nl,absorbeert water,rijst,bloemkool,rijst absorbeert water.,een bloemkool absorbeert water.,overlap,0.267606
7,nl,absorbeert water,rijst,aardappel,rijst absorbeert water.,een aardappel absorbeert water.,co-occurrence,0.728491
8,nl,absorbeert water,rijst,gelei,rijst absorbeert water.,gelei absorbeert water.,random,0.000000
9,nl,voegt lucht toe aan een mengsel,garde,blender,een garde voegt lucht toe aan een mengsel.,een blender voegt lucht toe aan een mengsel.,taxonomic,0.833333
10,nl,voegt lucht toe aan een mengsel,garde,spatel,een garde voegt lucht toe aan een mengsel.,een spatel voegt lucht toe aan een mengsel.,overlap,0.422535


In [ ]:
# https://www.sciencedirect.com/science/article/pii/S0001691814000985#ec0005
suppl1 = pd.read_excel('data/AoA_words/supplement1.xlsx') # old, all words present in 2
suppl2 = pd.read_excel('data/AoA_words/supplement2.xlsx') # combines ratings from different reserch

In [ ]:
suppl1.head()

,spelling,list,N,p.known,mean.rating,sd.rating
0,aai,1,18,1.000000,5.277778,2.492472
1,aalmoezenier,1,18,0.833333,11.800000,3.051931
2,aambei,1,18,0.944444,13.529412,2.831026
3,aanbaksel,1,18,1.000000,10.777778,2.734433
4,aanbellen,1,18,1.000000,7.388889,2.226548


In [ ]:
suppl2.head()

,Word,Translation,Ghyselinck2000,Unnamed: 3,Ghyselinck2003,Unnamed: 5,Moors2013,Unnamed: 7,Brysbaert2014,Unnamed: 9,Average,Unnamed: 11
0,NaN,NaN,AoA,%Known,AoA,%Known,AoA,%Known,AoA,%Known,AoA,%Known
1,mama,NaN,2.25,100,2,100,2.4,100,1.527027,100,2.044257,100
2,ja,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.25,100,2.25,100
3,papa,NaN,2.138889,100,2.2,100,2.5875,100,2.418919,100,2.336327,100
4,nee,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.555556,100,2.555556,100


In [ ]:
words_suppl1 = set(suppl1['spelling'].to_list())
len(words_suppl1), len(suppl1['spelling'])

(25886, 25888)

In [ ]:
words_suppl2 = set(suppl2['Word'].to_list())
len(words_suppl2), len(suppl2['Word'])

(31178, 31178)

In [ ]:
not_in_1 = words_suppl2 - words_suppl1
not_in_2 = words_suppl1 - words_suppl2
len(not_in_1), len(not_in_2)

(5292, 0)

In [ ]:
from simplemma import text_lemmatizer

data_xcomps = ' '.join(df_deduped['acceptable_sent'].to_list()) + ' ' + ' '.join(df_deduped['unacceptable_sent'].to_list())
vocab_xcomps = {text_lemmatizer(word.strip('.'), lang="nl")[0] : word for word in data_xcomps.split(" ")}
len(vocab_xcomps)

2953

In [ ]:
not_AoA = set(vocab_xcomps.keys()) - words_suppl2

In [ ]:
def sent_lemmas(sent):
    return {text_lemmatizer(w.strip('.'), lang="nl")[0] for w in sent.split(" ") if w.strip('.')}

uses_unknown = (
    df_deduped['acceptable_sent'].map(lambda s: bool(sent_lemmas(s) & not_AoA))
    | df_deduped['unacceptable_sent'].map(lambda s: bool(sent_lemmas(s) & not_AoA))
)

df_kids = df_deduped[~uses_unknown]
print(f'Dropped {uses_unknown.sum()} rows, {len(df_kids)} rows left')


Dropped 5523 rows, 8139 rows left


In [ ]:
df_kids['negative_sample_type'].value_counts()

negative_sample_type
random           2099
co-occurrence    2055
overlap          2033
taxonomic        1952
Name: count, dtype: int64

In [ ]:
if not Path('kids_NL_xcomps.csv').exists():
	df_kids.to_csv('kids_NL_xcomps.csv')